# Fixed-Height Study - Notebook 3 of 3: Run the Other Methods

## ▶ RUNTIME: **CPU only**
In Colab: **Runtime → Change runtime type → CPU**. Do not attach a GPU - nothing here uses it, and an attached GPU still uses up your GPU quota while it sits idle.

This notebook runs **FastME, FastTree and IQ-TREE** on the trees and sequences saved by Notebook 1 and pushes the results. Notebook 1 must have finished first. It does not depend on Notebook 2, so both can run at the same time.

**The three notebooks (run in this order)**

| # | Notebook | Runtime | What it does |
|---|---|---|---|
| 1 | `1_Generate_Trees_and_Sequences` | **CPU** | Simulates the trees and the sequences and saves them to the team repo |
| 2 | `2_Run_Phyloformer` | **GPU** | Runs Phyloformer on the saved data and pushes the results |
| 3 | `3_Run_Other_Methods` | **CPU** | Runs FastME, FastTree and IQ-TREE on the same saved data and pushes the results |

Notebook 1 must finish (and push) first. Notebooks 2 and 3 are independent of each other, so they can run at the same time in separate sessions.

## What this notebook does

For every grid cell it runs three methods on each saved alignment and compares the tree each one builds with the true tree.

| Method | How it is run |
|---|---|
| **FastME** | Distances computed from the alignment with the LG model, then FastME's NNI + SPR tree search (the same search used on Phyloformer's distances) |
| **FastTree** | `-lg -gamma` (LG model with gamma rate variation) |
| **IQ-TREE** | **LG data: `-m LG+GC`** (labelled `IQTree_LG+GC`). **Cherry data: `-m MFP`**, i.e. ModelFinder picks the best model (labelled `IQTree_MF`). This follows the paper: IQ-TREE's accuracy dropped a lot on Cherry data without ModelFinder. |

**Timing:** each method is timed **per tree**, wall-clock, using **one CPU thread**. File-format conversion is not counted. Do not run other heavy things in the same session while this runs, or the timings will be inflated.

**This notebook is slow, mostly because of IQ-TREE.** On one CPU thread, a 20-tip, 250-site alignment took about 20 seconds with `LG+GC` and about 40 seconds with ModelFinder in our tests; extrapolating from that, a 100-tip, 200-site alignment would take roughly 4 to 6 minutes. The time grows quickly with both the number of tips and the alignment length. This grid goes up to 250 tips and 1250 sites, so a single large alignment can take hours, and the whole grid takes far longer than a few sessions. FastME and FastTree take seconds. Because of this:
- Expect to need many sessions. Resume (below) picks up where the last one stopped.
- You can split the work across two sessions at the same time: set `MODELS = ["LG"]` in one and `MODELS = ["Cherry"]` in the other. They write to different files, so they do not clash.
- Two settings below limit the cost: a per-tree time limit, and an optional shorter list of models for ModelFinder. Trees that hit the limit are logged as failures with the reason `timeout`; mention this in the report if it happens.

**Duplicate sequences:** IQ-TREE is run with `-keep-ident`, so identical sequences stay in the tree (otherwise IQ-TREE drops them and the tree could not be compared with the true tree).


**What is in each results row (one row per tree)**

| Column | Meaning |
|---|---|
| `kf_score`, `rf`, `norm_rf`, `weighted_rf` | Kuhner-Felsenstein and Robinson-Foulds distances to the true tree |
| `mae` | Mean absolute error over all pairwise tip-to-tip distances (true vs inferred) |
| `runtime_seconds` | Time for **this one tree** (see each notebook for exactly what is timed) |
| `n_seqs`, `n_unique_sequences`, `n_duplicates` | Duplicate sequences in the alignment (`n_duplicates` = extra copies = `n_seqs - n_unique_sequences`) |
| `true_tree_height`, `true_total_length` | Size of the true tree (KF and MAE grow with it, so keep it for later normalising) |
| `seq_length_actual` | The alignment length that was really produced |

**Where things live in the team repo.** The code (setup files and notebooks) sits together in one folder. The data and the results go in their own folders, each with a `fixed_height_all_methods` subfolder, so earlier rounds are never touched.

```
fixed_height_all_methods/
├── setup/            the three helper files (setup_pipeline.py, fh_pipeline.py, pf_infer_timed.py)
└── notebooks/        these three notebooks

simulated_data/fixed_height_all_methods/
├── trees/true/clock_rate=R/tree_size=N/                          true trees         (notebook 1)
├── sequences/clock_rate=R/model=M/tree_size=N/seq_length=L/      alignments         (notebook 1)
└── trees/inferred/clock_rate=R/model=M/tree_size=N/seq_length=L/method=X/   trees built by each method (notebooks 2 and 3)

results/fixed_height_all_methods/
├── treesize_seqlength_rateR_{lg|cherry}_phyloformer_results.csv   (notebook 2)
├── treesize_seqlength_rateR_{lg|cherry}_baselines_results.csv     (notebook 3)
└── failures_rateR_{generate|phyloformer|baselines}.csv            anything that failed, and why
```

**If the session disconnects:** start a new session and run from the top. Cells already in the results file are skipped. Results are pushed after every tree size.

## 1. Settings

In [ ]:
import os
BASE_DIR = globals().get("BASE_DIR", os.getcwd())   # folder that will hold Phyloformer/ and team_repo/ (stays fixed if cells are re-run)

# ---- Settings: change ONLY CLOCK_RATE (use your own rate; same value in all three notebooks) ----
CLOCK_RATE = 0.1          # Siqi (0.1, 5), Deep (0.5, 4), Mihir (1, 3.5), Swetha (1.5, 3), Darren (2, 2.5)
TREE_HEIGHT = 1.0         # every tree is rescaled to this height BEFORE the clock rate is applied
MODELS = ["LG", "Cherry"] # substitution models the sequences were simulated under
TREE_SIZES = [50, 100, 150, 200, 250]       # number of tips
SEQ_LENGTHS = [250, 500, 750, 1000, 1250]     # alignment length
N_TREES = 20              # replicate trees per tree size

# ---- Settings specific to this notebook ----
IQTREE_TIMEOUT_S = 3600    # give up on one IQ-TREE run after this many seconds (logged as a 'timeout' failure)
MODELFINDER_MODELS = None  # None = ModelFinder tries everything (as in the paper).
                           # To speed it up you could use e.g. "LG,WAG,JTT" - but say so in the report.
N_WORKERS = 1              # trees run one at a time = clean timings. Raise to 2 to go faster
                           # (timings then include some sharing of the CPU - do not use for the final runs).

## 2. Setup
Checks the bundled programs and that this is a CPU runtime.

In [ ]:
# One-time setup. Prints a short checklist (details go to setup_log.txt).
# If it ends with "SETUP FAILED", fix the [FAIL] line and run this cell again.
import os
os.chdir(BASE_DIR)
!test -d team_repo || (git clone -q --filter=blob:none --sparse https://github.com/ACKarl/EPTI.git team_repo && git -C team_repo sparse-checkout set fixed_height_all_methods/setup)
!python team_repo/fixed_height_all_methods/setup/setup_pipeline.py --stage 3 --clock-rate {CLOCK_RATE}

In [ ]:
import os
import sys

PF_DIR = os.path.join(BASE_DIR, "Phyloformer")      # the tool being evaluated
TEAM_REPO = os.path.join(BASE_DIR, "team_repo")     # our shared repo (data and results are saved here)
sys.path.insert(0, os.path.join(TEAM_REPO, "fixed_height_all_methods", "setup"))
import fh_pipeline as fh                    # our helper functions (readable on GitHub: fixed_height_all_methods/setup/fh_pipeline.py)

WORK_DIR = os.path.join(PF_DIR, "data", "fh_work")   # scratch space for temporary files (wiped for every grid cell)

os.chdir(PF_DIR)                            # the repo's own scripts expect to run from this folder

## 3. GitHub login
Asked once, so the run is not interrupted later.

In [ ]:
# Asked ONCE per session. The token is kept in memory only (never written to a file or shown).
# It is used to push after every tree size, so a disconnect never costs more than one tree size.
creds = fh.ask_github_login(TEAM_REPO)

## 4. Sanity check (do this before the real run)

Runs all three programs once on one of the repo's own test alignments (about half a minute) and prints their scores. If any of them fails, fix that before starting.

In [ ]:
fh.sanity_check_baselines(PF_DIR, WORK_DIR)

## 5. Run the methods on the saved data

For each tree size, sequence length and model: skip the cell if it is already done, otherwise run the three methods, save the results, and log any failures. Pushes after every tree size.

In [ ]:
RDIR = fh.results_dir(TEAM_REPO)

for size in TREE_SIZES:
    print(f"=== tree size {size} ===")
    for seqlen in SEQ_LENGTHS:
        for model in MODELS:
            methods = fh.baseline_methods(model)    # FastME, FastTree, and IQ-TREE (LG+GC for LG, ModelFinder for Cherry)
            params = {"model": model, "clock_rate": CLOCK_RATE, "tree_height": TREE_HEIGHT,
                      "tree_size": size, "seq_length": seqlen}
            fname = fh.results_filename(CLOCK_RATE, model, "baselines")
            cell = {"model": model, "tree_size": size, "seq_length": seqlen}

            # Resume: skip cells whose results are already saved.
            if fh.cell_done(RDIR, fname, params, methods, N_TREES):
                print(f"  {model}, seq_length={seqlen}: already done, skipping")
                continue
            # The data comes from Notebook 1.
            if not fh.list_files(fh.sequences_dir(TEAM_REPO, CLOCK_RATE, model, size, seqlen), fh.ALN_EXTS):
                print(f"  {model}, seq_length={seqlen}: no saved sequences (run Notebook 1 first)")
                fh.save_failures(RDIR, CLOCK_RATE, "baselines", cell,
                                 [{"method": "-", "tree_id": "ALL", "reason": "no saved sequences"}])
                continue

            print(f"  {model}, seq_length={seqlen}: {', '.join(methods)}")
            try:
                df, fails = fh.run_baselines_cell(PF_DIR, TEAM_REPO, CLOCK_RATE, model, size, seqlen, N_TREES,
                                                  WORK_DIR, iqtree_timeout=IQTREE_TIMEOUT_S,
                                                  mf_models=MODELFINDER_MODELS, n_workers=N_WORKERS)
            except Exception as e:
                df, fails = None, [{"method": "-", "tree_id": "ALL", "reason": str(e)[:200]}]

            if df is not None:
                fh.save_results(df, params, fname, RDIR)
            fh.save_failures(RDIR, CLOCK_RATE, "baselines", cell, fails)

    try:
        fh.push_to_github(TEAM_REPO, fh.study_prefixes(CLOCK_RATE),
                          f"Baseline results: clock rate {CLOCK_RATE}, tree size {size}", creds)
    except Exception as e:
        print(f"  !! PUSH FAILED (results are safe locally, the final cell will try again): {str(e)[:200]}")

print("Finished.")

## 6. Check the results are complete
Lists any grid cell and method with fewer than 20 trees, and summarises the logged failures (including IQ-TREE timeouts).

In [ ]:
fh.completeness_report(TEAM_REPO, CLOCK_RATE, "baselines", MODELS, TREE_SIZES, SEQ_LENGTHS,
                       fh.baseline_methods, N_TREES)

## 7. Final push
Pushes anything not yet in the repo (safe to run even if everything is already pushed).

In [ ]:
fh.push_to_github(TEAM_REPO, fh.study_prefixes(CLOCK_RATE),
                  f"Baseline results: clock rate {CLOCK_RATE} (final)", creds)